# EyePACS Data Exploration\n\nThis notebook explores the EyePACS diabetic retinopathy dataset, validates image quality, creates stratified train/validation/test splits, saves report figures, and writes `reports/01_DATA_EXPLORATION_REPORT.txt`.\n\nExpected local dataset layout:\n\n```text\ndata/raw/eyepacs/trainLabels.csv\ndata/raw/eyepacs/train/*.jpeg\n```

## 1. Setup\n\nDownload the competition data with Kaggle CLI after authenticating with `kaggle auth login` or an API token. The notebook intentionally fails fast with a clear message if the dataset is not present.

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'dr_unified_v2' / 'dr_unified_v2'
LABELS_OUT = PROJECT_ROOT / 'data' / 'labels'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'

LABELS_OUT.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    0: 'No DR',
    1: 'Mild',
    2: 'Moderate',
    3: 'Severe',
    4: 'Proliferative DR',
}

sns.set_theme(style='whitegrid', palette='Set2')
random.seed(42)
np.random.seed(42)

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Missing dataset directory: {DATA_ROOT}')


## 2. Load and Inspect Data

In [ ]:
labels = pd.concat(
    [pd.read_csv(LABELS_OUT / f'{s}.csv') for s in ['train', 'val', 'test']],
    ignore_index=True,
)
labels = labels.rename(columns={'image_id': 'image'})
labels['image_path'] = labels['image_path'].apply(lambda p: str(PROJECT_ROOT / p))
labels['class_name'] = labels['severity'].map(CLASS_NAMES)

display(labels.head(10))
print('Shape:', labels.shape)
print('Columns:', labels.columns.tolist())
display(labels.dtypes.to_frame('dtype'))
display(labels.isna().sum().to_frame('missing_values'))

## 3. Class Distribution

In [ ]:
class_counts = labels['severity'].value_counts().sort_index()
class_percentages = (class_counts / len(labels) * 100).round(2)
distribution = pd.DataFrame({
    'severity': class_counts.index,
    'class_name': [CLASS_NAMES[i] for i in class_counts.index],
    'count': class_counts.values,
    'percentage': class_percentages.values,
})
display(distribution)

imbalance_ratio = class_counts.max() / class_counts.min()
dominant_class = int(class_counts.idxmax())
minority_class = int(class_counts.idxmin())
print(f'Imbalance ratio: {imbalance_ratio:.2f}x')
print(f'Dominant class: {dominant_class} ({CLASS_NAMES[dominant_class]})')
print(f'Minority class: {minority_class} ({CLASS_NAMES[minority_class]})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=distribution, x='severity', y='count', hue='class_name', ax=axes[0], dodge=False)
axes[0].set_title('EyePACS Class Distribution - Counts')
axes[0].set_xlabel('Severity Level')
axes[0].set_ylabel('Images')
axes[0].legend(title='Class')

sns.barplot(data=distribution, x='severity', y='percentage', hue='class_name', ax=axes[1], dodge=False)
axes[1].set_title('EyePACS Class Distribution - Percentage')
axes[1].set_xlabel('Severity Level')
axes[1].set_ylabel('Percent')
axes[1].legend(title='Class')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_class_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Image Statistics and Quality Checks

In [ ]:
labels['file_exists'] = labels['image_path'].apply(lambda path: Path(path).exists())
missing_files = labels.loc[~labels['file_exists'], 'image_path'].tolist()
print(f'Missing image files: {len(missing_files)}')

per_class = max(1, 500 // labels['severity'].nunique())
sample_df = pd.concat(
    [
        part.sample(min(len(part), per_class), random_state=42)
        for _, part in labels[labels['file_exists']].groupby('severity')
    ],
    ignore_index=True,
)
if len(sample_df) > 500:
    sample_df = sample_df.sample(500, random_state=42).reset_index(drop=True)

image_stats = []
corrupted_paths = []
for row in tqdm(sample_df.itertuples(index=False), total=len(sample_df), desc='Reading sample images'):
    image = cv2.imread(row.image_path)
    if image is None:
        corrupted_paths.append(row.image_path)
        continue
    height, width, channels = image.shape
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_stats.append({
        'image': row.image,
        'severity': row.severity,
        'class_name': row.class_name,
        'height': height,
        'width': width,
        'channels': channels,
        'format': Path(row.image_path).suffix.lower(),
        'is_rgb': rgb.shape[2] == 3,
        'image_path': row.image_path,
    })

image_stats_df = pd.DataFrame(image_stats)
display(image_stats_df.describe(include='all'))
print(f'Corrupted images in sample: {len(corrupted_paths)}')
print(f'Mean image size: {image_stats_df.height.mean():.0f}px x {image_stats_df.width.mean():.0f}px')
print(f'Formats found: {sorted(image_stats_df.format.dropna().unique().tolist())}')
print(f'RGB samples: {int(image_stats_df.is_rgb.sum())}/{len(image_stats_df)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=image_stats_df, x='width', y='height', hue='class_name', alpha=0.75, ax=ax)
ax.set_title('Sample Image Dimensions')
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_image_dimensions.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
gallery_rows = []
for severity in sorted(CLASS_NAMES):
    candidates = labels[(labels['severity'] == severity) & (labels['file_exists'])].sample(
        min(5, ((labels['severity'] == severity) & (labels['file_exists'])).sum()), random_state=42
    )
    gallery_rows.extend(candidates.to_dict('records'))

fig, axes = plt.subplots(5, 5, figsize=(15, 15))
for ax in axes.flat:
    ax.axis('off')

for idx, row in enumerate(gallery_rows):
    ax = axes[row['severity'], idx % 5]
    image = cv2.imread(row['image_path'])
    if image is None:
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    ax.imshow(image)
    ax.set_title(f"{row['severity']}: {row['class_name']}", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_sample_images_per_class.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Full Data Quality Report

In [ ]:
invalid_labels = labels.loc[~labels['severity'].isin(CLASS_NAMES.keys())]
null_values = int(labels[['image', 'severity']].isna().sum().sum())

readability_records = []
for row in tqdm(labels[labels['file_exists']].itertuples(index=False), total=int(labels['file_exists'].sum()), desc='Checking readable images'):
    image = cv2.imread(row.image_path)
    readability_records.append({'image': row.image, 'readable': image is not None})

readability_df = pd.DataFrame(readability_records)
readability_map = dict(zip(readability_df['image'], readability_df['readable'])) if not readability_df.empty else {}
labels['readable'] = labels['image'].map(readability_map).fillna(False)

duplicate_ids = int(labels['image'].duplicated().sum())
import os
from collections import defaultdict

# Two-stage duplicate check: group by file size first, hash only candidates
size_groups = defaultdict(list)
for path in labels.loc[labels['readable'], 'image_path']:
    try:
        size_groups[os.path.getsize(path)].append(path)
    except OSError:
        pass

duplicate_hashes = []
for size, paths in size_groups.items():
    if len(paths) >= 2:
        seen_hashes = set()
        for p in paths:
            try:
                with open(p, 'rb') as handle:
                    digest = hashlib.md5(handle.read()).hexdigest()
                if digest in seen_hashes:
                    duplicate_hashes.append(digest)
                else:
                    seen_hashes.add(digest)
            except OSError:
                pass

valid_images = int(labels['readable'].sum())
corrupted_or_missing = len(labels) - valid_images
quality_score = valid_images / len(labels) * 100

quality_report = {
    'total_labeled_images': len(labels),
    'valid_readable_images': valid_images,
    'corrupted_or_missing': corrupted_or_missing,
    'missing_files': len(missing_files),
    'invalid_labels': len(invalid_labels),
    'null_values': null_values,
    'duplicate_image_ids': duplicate_ids,
    'duplicate_image_hashes': len(duplicate_hashes),
    'quality_score': round(quality_score, 2),
}
print(json.dumps(quality_report, indent=2))

## 6. Stratified Train/Validation/Test Splits

In [ ]:
# Load pre-split stratified datasets without overwriting
train_df = pd.read_csv(LABELS_OUT / 'train.csv')
val_df = pd.read_csv(LABELS_OUT / 'val.csv')
test_df = pd.read_csv(LABELS_OUT / 'test.csv')
usable = pd.concat([train_df, val_df, test_df], ignore_index=True)

for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name}: {len(split_df):,} images')
    split_counts = split_df['severity'].value_counts(normalize=False).sort_index()
    split_percent = split_df['severity'].value_counts(normalize=True).sort_index() * 100
    for severity in sorted(CLASS_NAMES):
        print(f'  Class {severity}: {split_counts.get(severity, 0):,} ({split_percent.get(severity, 0):.2f}%)')


## 7. Imbalance Analysis and Class Weights

In [ ]:
classes = np.array(sorted(CLASS_NAMES))
weights = compute_class_weight(class_weight='balanced', classes=classes, y=usable['severity'])
class_weights = dict(zip(classes.tolist(), weights.round(4).tolist()))
imbalance_details = distribution.copy()
imbalance_details['ratio_vs_minority'] = (imbalance_details['count'] / class_counts.min()).round(2)
imbalance_details['class_weight'] = imbalance_details['severity'].map(class_weights)
display(imbalance_details)

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(distribution['count'], labels=distribution['class_name'], autopct='%1.1f%%', startangle=90)
ax.set_title('EyePACS Class Imbalance')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_class_imbalance_analysis.png', dpi=200, bbox_inches='tight')
plt.show()

print('Class weights for weighted loss:', class_weights)

## 8. Write Final Report

In [ ]:
height_mean = image_stats_df['height'].mean()
width_mean = image_stats_df['width'].mean()
height_min, height_max = image_stats_df['height'].min(), image_stats_df['height'].max()
width_min, width_max = image_stats_df['width'].min(), image_stats_df['width'].max()

def class_line(severity):
    return f"  {severity} ({CLASS_NAMES[severity]}): {class_counts.get(severity, 0):,} images ({class_percentages.get(severity, 0):.2f}%)"

report = f'''========================================
EYEPACS DATASET EXPLORATION REPORT
========================================

DATASET OVERVIEW:
- Total images: {len(labels):,}
- Number of classes: 5
- Image format: JPEG
- Target size for ML: 224x224

CLASS DISTRIBUTION:
{chr(10).join(class_line(severity) for severity in sorted(CLASS_NAMES))}

IMBALANCE ANALYSIS:
- Imbalance ratio (max/min): {imbalance_ratio:.2f}x
- Dominant class: {dominant_class} ({CLASS_NAMES[dominant_class]}) ({class_percentages[dominant_class]:.2f}% of data)
- Minority class: {minority_class} ({CLASS_NAMES[minority_class]}) ({class_percentages[minority_class]:.2f}% of data)
- STATUS: HIGH IMBALANCE - Requires special handling
- Class weights: {class_weights}

IMAGE STATISTICS:
- Mean height: {height_mean:.0f}px
- Mean width: {width_mean:.0f}px
- Height range: {height_min:.0f}-{height_max:.0f}px
- Width range: {width_min:.0f}-{width_max:.0f}px
- Color channels: 3 (RGB)

DATA QUALITY:
- Valid images: {quality_score:.2f}%
- Corrupted images: {corrupted_or_missing - len(missing_files):,}
- Missing files: {len(missing_files):,}
- Duplicate image IDs: {duplicate_ids:,}
- Duplicate image hashes: {len(duplicate_hashes):,}
- Null values: {null_values:,}
- Invalid labels: {len(invalid_labels):,}

TRAIN/VAL/TEST SPLITS:
- Training:   {len(train_df):,} images ({len(train_df) / len(usable) * 100:.2f}%)
- Validation: {len(val_df):,} images ({len(val_df) / len(usable) * 100:.2f}%)
- Test:       {len(test_df):,} images ({len(test_df) / len(usable) * 100:.2f}%)
- Stratification: Verified

KEY FINDINGS:
1. Severe class imbalance - Class 0 dominates
2. Need weighted loss or focal loss for training
3. Focus on Recall for high-risk classes (3, 4)
4. Image sizes vary - need standardization
5. Readability and missing-file checks completed before split generation

NEXT STEPS (WEEK 2):
1. Image preprocessing pipeline
2. Data augmentation strategy
3. Transfer learning baseline (EfficientNet)
4. Handle class imbalance with weighted loss
5. Define evaluation metrics (Macro-F1, per-class Recall)
'''

report_path = REPORTS_DIR / '01_DATA_EXPLORATION_REPORT.txt'
report_path.write_text(report, encoding='utf-8')
print(report)
print(f'Report saved to {report_path}')